# No Permutation - Optuna Tuning || TSTR Evaluation || || User higher number of trials for actual evaluation

In [ ]:
import os, gc, random, warnings
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")


import numpy as np
import pandas as pd
import torch
import optuna


from sklearn.model_selection import StratifiedShuffleSplit, RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.utils import check_random_state
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from bstabdiff import BSTabDiff


GLOBAL_SEED = 42
CV_SEED = 0
N_TRIALS = 5
INNER_VAL_SIZE = 0.2
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
CSV_PATH = "coloncancer_encoded.csv"
LABEL_COL = "label"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_data():
    df = pd.read_csv(CSV_PATH)
    y_raw = df[LABEL_COL].values
    X_df = df.drop(columns=[LABEL_COL]).select_dtypes(include=[np.number])
    X = np.nan_to_num(X_df.to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    if y_raw.dtype.kind in {"U", "S", "O"}:
        y, _ = pd.factorize(y_raw)
    else:
        u = np.unique(y_raw)
        y, _ = pd.factorize(y_raw) if not np.array_equal(u, np.arange(len(u))) else (y_raw.astype(int), None)
    return X, np.asarray(y, dtype=int)


def make_lr():
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
        ("clf", LogisticRegression(max_iter=10000, class_weight="balanced", solver="lbfgs"))
    ])


def safe_auc_or_acc(clf, X, y):
    try:
        return float(roc_auc_score(y, clf.predict_proba(X)[:, 1]))
    except Exception:
        return float(accuracy_score(y, clf.predict(X)))


X, y = load_data()
assert X.shape == (62, 2000)
set_seed(GLOBAL_SEED)


splitter = StratifiedShuffleSplit(n_splits=1, test_size=INNER_VAL_SIZE, random_state=GLOBAL_SEED)
(itr, iva), = splitter.split(X, y)
X_inner_tr, y_inner_tr = X[itr], y[itr]
X_inner_va, y_inner_va = X[iva], y[iva]


def objective(trial):
    params = {
        "M": trial.suggest_categorical("M", [8, 12, 16, 24, 32, 48, 64]),
        "prior_type": trial.suggest_categorical("prior_type", ["diffusion", "flow"]),
        "prior_epochs": trial.suggest_categorical("prior_epochs", [300, 500, 800, 1200, 1600]),
        "prior_batch": trial.suggest_categorical("prior_batch", [16, 32, 64, 128]),
        "prior_lr": trial.suggest_float("prior_lr", 5e-5, 5e-3, log=True),
        "ema_decay": trial.suggest_float("ema_decay", 0.99, 0.9999, log=True),
        "n_syn": trial.suggest_categorical("n_syn", [64, 100, 128, 200, 256])
    }
    trial_seed = GLOBAL_SEED + 1000 + trial.number
    try:
        model = BSTabDiff(
            feature_specs=None, M=params["M"], blocks=None, permute_features=False,
            prior_type=params["prior_type"], device=DEVICE, random_state=trial_seed,
            prior_epochs=params["prior_epochs"], prior_batch=params["prior_batch"],
            prior_lr=params["prior_lr"], verbose_every=0, save_dir=None,
            save_name=f"colon_noperm_trial_{trial.number}", save_best=True,
            use_ema=True, ema_decay=params["ema_decay"],
            use_gobs=False, use_gobs_fc=False
        )
        model.fit(X_inner_tr, y_inner_tr)
        X_syn, _, y_syn = model.sample(n_samples=params["n_syn"], return_mask=True)
        X_syn = np.nan_to_num(np.asarray(X_syn, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        y_syn = np.asarray(y_syn, dtype=int)
        if np.unique(y_syn).size < 2:
            return 0.0
        clf = make_lr()
        clf.fit(X_syn, y_syn)
        score = safe_auc_or_acc(clf, X_inner_va, y_inner_va)
        trial.set_user_attr("raw_validation_score", score)
        return score - 1e-4 * params["M"]
    except Exception as e:
        trial.set_user_attr("error", repr(e))
        return 0.0
    finally:
        cleanup()


study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=GLOBAL_SEED, multivariate=True, warn_independent_sampling=False),
    pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=0)
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)


best = dict(study.best_params)
print("\nBest Optuna objective:", study.best_value)
print("Best parameters:", best)


def final_eval(best):
    rng = check_random_state(CV_SEED)
    rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=CV_SEED)
    tstr_acc, tstr_auc, trtr_acc, trtr_auc = [], [], [], []
    for fold, (tr, te) in enumerate(rkf.split(X, y), 1):
        Xtr, ytr, Xte, yte = X[tr], y[tr], X[te], y[te]
        fold_seed = int(rng.randint(0, 2**31 - 1))
        model = BSTabDiff(
            feature_specs=None, M=best["M"], blocks=None, permute_features=False,
            prior_type=best["prior_type"], device=DEVICE, random_state=fold_seed,
            prior_epochs=best["prior_epochs"], prior_batch=best["prior_batch"],
            prior_lr=best["prior_lr"], verbose_every=0, save_dir=None,
            save_name="colon_noperm_best", save_best=True, use_ema=True,
            ema_decay=best["ema_decay"], use_gobs=False, use_gobs_fc=False
        )
        model.fit(Xtr, ytr)
        X_syn, _, y_syn = model.sample(n_samples=best["n_syn"], return_mask=True)


        clf = make_lr()
        clf.fit(X_syn, y_syn)
        p = clf.predict_proba(Xte)[:, 1]
        tstr_acc.append(accuracy_score(yte, (p >= 0.5).astype(int)))
        tstr_auc.append(roc_auc_score(yte, p))


        clf_real = make_lr()
        clf_real.fit(Xtr, ytr)
        p_real = clf_real.predict_proba(Xte)[:, 1]
        trtr_acc.append(accuracy_score(yte, (p_real >= 0.5).astype(int)))
        trtr_auc.append(roc_auc_score(yte, p_real))


        print(f"Fold {fold:02d}/25 | TSTR ACC={tstr_acc[-1]:.4f} AUC={tstr_auc[-1]:.4f}")
        del model, clf, clf_real
        cleanup()


    print("\nTUNED NO PERMUTATION — STRICT 5x5")
    print(f"TSTR ACC = {np.mean(tstr_acc):.4f} ± {np.std(tstr_acc):.4f}")
    print(f"TSTR AUC = {np.mean(tstr_auc):.4f} ± {np.std(tstr_auc):.4f}")
    print(f"TRTR ACC = {np.mean(trtr_acc):.4f} ± {np.std(trtr_acc):.4f}")
    print(f"TRTR AUC = {np.mean(trtr_auc):.4f} ± {np.std(trtr_auc):.4f}")


final_eval(best)
study.trials_dataframe().to_csv("colon_optuna_noperm_5trials.csv", index=False)

[I 2026-09-30 04:21:26,337] A new study created in memory with name: no-name-651de1c0-dac6-4e85-8663-9476813bcb24


  0%|          | 0/5 [00:00<?, ?it/s]

[I 2026-09-30 04:21:32,750] Trial 0 finished with value: 0.9488 and parameters: {'M': 12, 'prior_type': 'diffusion', 'prior_epochs': 800, 'prior_batch': 128, 'prior_lr': 0.0003654769917956456, 'ema_decay': 0.9928730087657603, 'n_syn': 64}. Best is trial 0 with value: 0.9488.
[I 2026-09-30 04:21:41,689] Trial 1 finished with value: 0.9491999999999999 and parameters: {'M': 8, 'prior_type': 'flow', 'prior_epochs': 300, 'prior_batch': 64, 'prior_lr': 0.0032931446587915556, 'ema_decay': 0.9925524818123974, 'n_syn': 64}. Best is trial 1 with value: 0.9491999999999999.
[I 2026-09-30 04:21:49,413] Trial 2 finished with value: 0.9491999999999999 and parameters: {'M': 8, 'prior_type': 'diffusion', 'prior_epochs': 1200, 'prior_batch': 128, 'prior_lr': 7.048087574907931e-05, 'ema_decay': 0.9997695422398135, 'n_syn': 200}. Best is trial 1 with value: 0.9491999999999999.
[I 2026-09-30 04:21:59,128] Trial 3 finished with value: 0.9451999999999999 and parameters: {'M': 48, 'prior_type': 'diffusion', '

# GO-BS + Optuna Tuning || TSTR Evaluation || || User higher number of trials for actual evaluation

In [ ]:
import os, gc, random, warnings
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")


import numpy as np
import pandas as pd
import torch
import optuna


from sklearn.model_selection import StratifiedShuffleSplit, RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.utils import check_random_state
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from bstabdiff import BSTabDiff


GLOBAL_SEED = 42
CV_SEED = 0
N_TRIALS = 5
INNER_VAL_SIZE = 0.2
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
CSV_PATH = "coloncancer_encoded.csv"
LABEL_COL = "label"


FIXED = {
    "M": 8, "prior_type": "flow", "prior_epochs": 300,
    "prior_batch": 64, "prior_lr": 3.2931e-3,
    "ema_decay": 0.9926, "n_syn": 64
}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_data():
    df = pd.read_csv(CSV_PATH)
    y_raw = df[LABEL_COL].values
    X = np.nan_to_num(
        df.drop(columns=[LABEL_COL]).select_dtypes(include=[np.number]).to_numpy(dtype=np.float32),
        nan=0.0, posinf=0.0, neginf=0.0
    )
    if y_raw.dtype.kind in {"U", "S", "O"}:
        y, _ = pd.factorize(y_raw)
    else:
        u = np.unique(y_raw)
        y, _ = pd.factorize(y_raw) if not np.array_equal(u, np.arange(len(u))) else (y_raw.astype(int), None)
    return X, np.asarray(y, dtype=int)


def make_lr():
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
        ("clf", LogisticRegression(max_iter=10000, class_weight="balanced", solver="lbfgs"))
    ])


def safe_auc_or_acc(clf, X, y):
    try:
        return float(roc_auc_score(y, clf.predict_proba(X)[:, 1]))
    except Exception:
        return float(accuracy_score(y, clf.predict(X)))


X, y = load_data()
assert X.shape == (62, 2000)
set_seed(GLOBAL_SEED)


splitter = StratifiedShuffleSplit(n_splits=1, test_size=INNER_VAL_SIZE, random_state=GLOBAL_SEED)
(itr, iva), = splitter.split(X, y)
X_inner_tr, y_inner_tr = X[itr], y[itr]
X_inner_va, y_inner_va = X[iva], y[iva]


def sample_gobs(trial):
    return {
        "gobs_num_clusters": trial.suggest_categorical("gobs_num_clusters", [3, 4, 5, 6, 7, 8, 10, 12]),
        "gobs_metric": trial.suggest_categorical("gobs_metric", ["correlation", "cosine", "manhattan", "euclidean", "kl_divergence"]),
        "gobs_bins": trial.suggest_categorical("gobs_bins", [16, 24, 32, 48, 64]),
        "gobs_top_k": trial.suggest_categorical("gobs_top_k", [None, 16, 32, 64, 128, 256]),
        "gobs_refine_order": trial.suggest_categorical("gobs_refine_order", [True, False]),
        "gobs_direction_select": trial.suggest_categorical("gobs_direction_select", [True, False]),
        "gobs_refine_passes": trial.suggest_categorical("gobs_refine_passes", [0, 1, 2, 3]),
        "gobs_boundary_refine_passes": trial.suggest_categorical("gobs_boundary_refine_passes", [0, 1, 3, 5, 8]),
        "gobs_boundary_window": trial.suggest_categorical("gobs_boundary_window", [4, 8, 12, 16, 24]),
        "gobs_lambda_cross": trial.suggest_float("gobs_lambda_cross", 0.1, 5.0, log=True),
        "gobs_gamma_balance": trial.suggest_float("gobs_gamma_balance", 1e-8, 1e-2, log=True)
    }


def build_model(g, seed, name):
    return BSTabDiff(
        feature_specs=None, M=FIXED["M"], blocks=None, permute_features=False,
        prior_type=FIXED["prior_type"], device=DEVICE, random_state=seed,
        prior_epochs=FIXED["prior_epochs"], prior_batch=FIXED["prior_batch"],
        prior_lr=FIXED["prior_lr"], verbose_every=0, save_dir=None,
        save_name=name, save_best=True, use_ema=True, ema_decay=FIXED["ema_decay"],
        use_gobs=True, use_gobs_fc=False, gobs_use_cpu_kmeans=False, **g
    )


def objective(trial):
    g = sample_gobs(trial)
    trial_seed = GLOBAL_SEED + 1000 + trial.number
    try:
        model = build_model(g, trial_seed, f"colon_gobs_trial_{trial.number}")
        model.fit(X_inner_tr, y_inner_tr)
        X_syn, _, y_syn = model.sample(n_samples=FIXED["n_syn"], return_mask=True)
        X_syn = np.nan_to_num(np.asarray(X_syn, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        y_syn = np.asarray(y_syn, dtype=int)
        if np.unique(y_syn).size < 2:
            return 0.0
        clf = make_lr()
        clf.fit(X_syn, y_syn)
        score = safe_auc_or_acc(clf, X_inner_va, y_inner_va)
        penalty = 1e-4 * g["gobs_num_clusters"]
        if g["gobs_top_k"] is not None:
            penalty += 1e-7 * g["gobs_top_k"]
        trial.set_user_attr("raw_validation_score", score)
        return score - penalty
    except Exception as e:
        trial.set_user_attr("error", repr(e))
        return 0.0
    finally:
        cleanup()


study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=GLOBAL_SEED, multivariate=True, group=True, warn_independent_sampling=False),
    pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=0)
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)


best = dict(study.best_params)
print("\nBest Optuna objective:", study.best_value)
print("Best GO-BS parameters:", best)


def final_eval(best):
    rng = check_random_state(CV_SEED)
    rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=CV_SEED)
    ta, tu, ra, ru = [], [], [], []
    for fold, (tr, te) in enumerate(rkf.split(X, y), 1):
        Xtr, ytr, Xte, yte = X[tr], y[tr], X[te], y[te]
        fold_seed = int(rng.randint(0, 2**31 - 1))
        model = build_model(best, fold_seed, "colon_gobs_best")
        model.fit(Xtr, ytr)
        X_syn, _, y_syn = model.sample(n_samples=FIXED["n_syn"], return_mask=True)


        clf = make_lr()
        clf.fit(X_syn, y_syn)
        p = clf.predict_proba(Xte)[:, 1]
        ta.append(accuracy_score(yte, (p >= 0.5).astype(int)))
        tu.append(roc_auc_score(yte, p))


        clf_real = make_lr()
        clf_real.fit(Xtr, ytr)
        pr = clf_real.predict_proba(Xte)[:, 1]
        ra.append(accuracy_score(yte, (pr >= 0.5).astype(int)))
        ru.append(roc_auc_score(yte, pr))


        print(f"Fold {fold:02d}/25 | TSTR ACC={ta[-1]:.4f} AUC={tu[-1]:.4f}")
        del model, clf, clf_real
        cleanup()


    print("\nGO-BS — STRICT 5x5")
    print(f"TSTR ACC = {np.mean(ta):.4f} ± {np.std(ta):.4f}")
    print(f"TSTR AUC = {np.mean(tu):.4f} ± {np.std(tu):.4f}")
    print(f"TRTR ACC = {np.mean(ra):.4f} ± {np.std(ra):.4f}")
    print(f"TRTR AUC = {np.mean(ru):.4f} ± {np.std(ru):.4f}")


final_eval(best)
study.trials_dataframe().to_csv("colon_optuna_gobs_5trials.csv", index=False)

[I 2026-09-30 04:28:58,174] A new study created in memory with name: no-name-09c12707-bf1c-495b-a702-61aaea5f664e


  0%|          | 0/5 [00:00<?, ?it/s]

[I 2026-09-30 04:29:09,464] Trial 0 finished with value: 0.9495967999999999 and parameters: {'gobs_num_clusters': 4, 'gobs_metric': 'euclidean', 'gobs_bins': 64, 'gobs_top_k': 32, 'gobs_refine_order': False, 'gobs_direction_select': False, 'gobs_refine_passes': 2, 'gobs_boundary_refine_passes': 3, 'gobs_boundary_window': 8, 'gobs_lambda_cross': 0.11439974749291269, 'gobs_gamma_balance': 0.0028570800750407216}. Best is trial 0 with value: 0.9495967999999999.
[I 2026-09-30 04:29:22,053] Trial 1 finished with value: 0.949 and parameters: {'gobs_num_clusters': 10, 'gobs_metric': 'correlation', 'gobs_bins': 48, 'gobs_top_k': None, 'gobs_refine_order': False, 'gobs_direction_select': True, 'gobs_refine_passes': 1, 'gobs_boundary_refine_passes': 8, 'gobs_boundary_window': 4, 'gobs_lambda_cross': 1.7361170890393298, 'gobs_gamma_balance': 6.688747907702057e-05}. Best is trial 0 with value: 0.9495967999999999.
[I 2026-09-30 04:29:34,680] Trial 2 finished with value: 0.9246984 and parameters: {'g

# GO-BS-FC + Optuna Tuning || TSTR Evaluation || User higher number of trials for actual evaluation

In [ ]:
import os, gc, random, warnings
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")


import numpy as np
import pandas as pd
import torch
import optuna


from sklearn.model_selection import StratifiedShuffleSplit, RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.utils import check_random_state
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from bstabdiff import BSTabDiff


GLOBAL_SEED = 42
CV_SEED = 0
N_TRIALS = 5
INNER_VAL_SIZE = 0.2
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
CSV_PATH = "coloncancer_encoded.csv"
LABEL_COL = "label"


FIXED = {
    "M": 8, "prior_type": "flow", "prior_epochs": 300,
    "prior_batch": 64, "prior_lr": 3.2931e-3,
    "ema_decay": 0.9926, "n_syn": 64
}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_data():
    df = pd.read_csv(CSV_PATH)
    y_raw = df[LABEL_COL].values
    X = np.nan_to_num(
        df.drop(columns=[LABEL_COL]).select_dtypes(include=[np.number]).to_numpy(dtype=np.float32),
        nan=0.0, posinf=0.0, neginf=0.0
    )
    if y_raw.dtype.kind in {"U", "S", "O"}:
        y, _ = pd.factorize(y_raw)
    else:
        u = np.unique(y_raw)
        y, _ = pd.factorize(y_raw) if not np.array_equal(u, np.arange(len(u))) else (y_raw.astype(int), None)
    return X, np.asarray(y, dtype=int)


def make_lr():
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
        ("clf", LogisticRegression(max_iter=10000, class_weight="balanced", solver="lbfgs"))
    ])


def safe_auc_or_acc(clf, X, y):
    try:
        return float(roc_auc_score(y, clf.predict_proba(X)[:, 1]))
    except Exception:
        return float(accuracy_score(y, clf.predict(X)))


X, y = load_data()
assert X.shape == (62, 2000)
set_seed(GLOBAL_SEED)


splitter = StratifiedShuffleSplit(n_splits=1, test_size=INNER_VAL_SIZE, random_state=GLOBAL_SEED)
(itr, iva), = splitter.split(X, y)
X_inner_tr, y_inner_tr = X[itr], y[itr]
X_inner_va, y_inner_va = X[iva], y[iva]


def sample_gobs_fc(trial):
    return {
        "gobs_num_clusters": trial.suggest_int("gobs_num_clusters", 3, 10),
        "gobs_metric": trial.suggest_categorical("gobs_metric", ["correlation", "cosine", "euclidean", "manhattan"]),
        "gobs_bins": trial.suggest_categorical("gobs_bins", [16, 32, 64]),
        "gobs_top_k": trial.suggest_categorical("gobs_top_k", [None, 8, 16, 32, 64]),
        "gobs_refine_order": trial.suggest_categorical("gobs_refine_order", [True, False]),
        "gobs_direction_select": trial.suggest_categorical("gobs_direction_select", [True, False]),
        "gobs_refine_passes": trial.suggest_int("gobs_refine_passes", 0, 3),
        "gobs_boundary_refine_passes": trial.suggest_int("gobs_boundary_refine_passes", 1, 8),
        "gobs_boundary_window": trial.suggest_categorical("gobs_boundary_window", [4, 8, 12, 16]),
        "gobs_lambda_cross": trial.suggest_float("gobs_lambda_cross", 0.1, 5.0, log=True),
        "gobs_gamma_balance": trial.suggest_float("gobs_gamma_balance", 1e-6, 1e-1, log=True)
    }


def build_model(g, seed, name):
    return BSTabDiff(
        feature_specs=None, M=FIXED["M"], blocks=None, permute_features=False,
        prior_type=FIXED["prior_type"], device=DEVICE, random_state=seed,
        prior_epochs=FIXED["prior_epochs"], prior_batch=FIXED["prior_batch"],
        prior_lr=FIXED["prior_lr"], verbose_every=0, save_dir=None,
        save_name=name, save_best=True, use_ema=True, ema_decay=FIXED["ema_decay"],
        use_gobs=True, use_gobs_fc=True, gobs_use_cpu_kmeans=False, **g
    )


def objective(trial):
    g = sample_gobs_fc(trial)
    trial_seed = GLOBAL_SEED + 1000 + trial.number
    try:
        model = build_model(g, trial_seed, f"colon_gobs_fc_trial_{trial.number}")
        model.fit(X_inner_tr, y_inner_tr)
        X_syn, _, y_syn = model.sample(n_samples=FIXED["n_syn"], return_mask=True)
        X_syn = np.nan_to_num(np.asarray(X_syn, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        y_syn = np.asarray(y_syn, dtype=int)
        if np.unique(y_syn).size < 2:
            return 0.0
        clf = make_lr()
        clf.fit(X_syn, y_syn)
        score = safe_auc_or_acc(clf, X_inner_va, y_inner_va)
        trial.set_user_attr("raw_validation_score", score)
        return score - 1e-4 * FIXED["M"]
    except Exception as e:
        trial.set_user_attr("error", repr(e))
        return 0.0
    finally:
        cleanup()


study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=GLOBAL_SEED, multivariate=True, warn_independent_sampling=False),
    pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=0)
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)


best = dict(study.best_params)
print("\nBest Optuna objective:", study.best_value)
print("Best GO-BS-FC parameters:", best)


def final_eval(best):
    rng = check_random_state(CV_SEED)
    rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=CV_SEED)
    ta, tu, ra, ru = [], [], [], []
    for fold, (tr, te) in enumerate(rkf.split(X, y), 1):
        Xtr, ytr, Xte, yte = X[tr], y[tr], X[te], y[te]
        fold_seed = int(rng.randint(0, 2**31 - 1))
        model = build_model(best, fold_seed, "colon_gobs_fc_best")
        model.fit(Xtr, ytr)
        X_syn, _, y_syn = model.sample(n_samples=FIXED["n_syn"], return_mask=True)


        clf = make_lr()
        clf.fit(X_syn, y_syn)
        p = clf.predict_proba(Xte)[:, 1]
        ta.append(accuracy_score(yte, (p >= 0.5).astype(int)))
        tu.append(roc_auc_score(yte, p))


        clf_real = make_lr()
        clf_real.fit(Xtr, ytr)
        pr = clf_real.predict_proba(Xte)[:, 1]
        ra.append(accuracy_score(yte, (pr >= 0.5).astype(int)))
        ru.append(roc_auc_score(yte, pr))


        print(f"Fold {fold:02d}/25 | TSTR ACC={ta[-1]:.4f} AUC={tu[-1]:.4f}")
        del model, clf, clf_real
        cleanup()


    print("\nGO-BS-FC — STRICT 5x5")
    print(f"TSTR ACC = {np.mean(ta):.4f} ± {np.std(ta):.4f}")
    print(f"TSTR AUC = {np.mean(tu):.4f} ± {np.std(tu):.4f}")
    print(f"TRTR ACC = {np.mean(ra):.4f} ± {np.std(ra):.4f}")
    print(f"TRTR AUC = {np.mean(ru):.4f} ± {np.std(ru):.4f}")


final_eval(best)
study.trials_dataframe().to_csv("colon_optuna_gobs_fc_5trials.csv", index=False)

[I 2026-09-30 04:36:51,731] A new study created in memory with name: no-name-3d4ac95c-85ca-447e-8a78-778e9585b9fb


  0%|          | 0/5 [00:00<?, ?it/s]

[I 2026-09-30 04:37:04,904] Trial 0 finished with value: 0.9742 and parameters: {'gobs_num_clusters': 5, 'gobs_metric': 'correlation', 'gobs_bins': 64, 'gobs_top_k': 32, 'gobs_refine_order': True, 'gobs_direction_select': False, 'gobs_refine_passes': 2, 'gobs_boundary_refine_passes': 4, 'gobs_boundary_window': 8, 'gobs_lambda_cross': 0.41921593504109744, 'gobs_gamma_balance': 0.00019069966103000456}. Best is trial 0 with value: 0.9742.
[I 2026-09-30 04:37:19,124] Trial 1 finished with value: 0.9491999999999999 and parameters: {'gobs_num_clusters': 9, 'gobs_metric': 'euclidean', 'gobs_bins': 16, 'gobs_top_k': 8, 'gobs_refine_order': True, 'gobs_direction_select': False, 'gobs_refine_passes': 0, 'gobs_boundary_refine_passes': 8, 'gobs_boundary_window': 8, 'gobs_lambda_cross': 0.8488762161408708, 'gobs_gamma_balance': 8.399864445957506e-06}. Best is trial 0 with value: 0.9742.
[I 2026-09-30 04:37:31,804] Trial 2 finished with value: 0.9491999999999999 and parameters: {'gobs_num_clusters':